# Experiment 7: Clustering using Python
## K-Means, Agglomerative and DBSCAN — Airline Passenger Satisfaction

**Aim:** Group airline passengers into clusters using six numeric features, without using the satisfaction label, and compare the three clustering algorithms.

**How to run**
- **Google Colab:** run the cells in order. When the load cell asks, choose `train.csv`.
- **Windows Jupyter:** the load cell looks for `train.csv` in `DATA_DIR` and then in the notebook's own folder. Edit `DATA_DIR` if your file is somewhere else.
- Run every cell **once, top to bottom**, and take each screenshot straight after its cell runs. Do not re-run cells later just to take a screenshot.
- The cell that chooses *k* may take about a minute.

**Screenshots required (8):** 1 = features and scaling, 2 = choosing k, 3 = K-Means result, 4 = dendrogram, 5 = Agglomerative result, 6 = DBSCAN k-distance plot, 7 = DBSCAN result, 8 = comparison. The other cells need no screenshot.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
print('Libraries imported successfully.')

## Load the dataset
**Colab:** choose `train.csv`. **Windows Jupyter:** edit `DATA_DIR` if needed. No screenshot needed.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import files
    print('Select train.csv in the file picker.')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No file was uploaded. Run this cell again and choose train.csv.')
    train_path = Path(list(uploaded.keys())[0])
else:
    DATA_DIR = Path(r'C:\Users\Admin\Downloads\archive (1)')
    found = [p for p in (DATA_DIR / 'train.csv', Path('train.csv')) if p.is_file()]
    if not found:
        raise FileNotFoundError(f'train.csv not found in {DATA_DIR} or in the current folder. Edit DATA_DIR above.')
    train_path = found[0]

df = pd.read_csv(train_path)
print('Loaded:', train_path, '| shape:', df.shape)

## Select features, scale and reduce to 2-D — Screenshot 1
Clustering uses distances, so only numeric columns are used and every column is standardised (mean 0, standard deviation 1). The satisfaction label is **not** used. PCA is used only to draw the clusters on a 2-D chart.

In [ ]:
features = ['Age', 'Flight Distance', 'Inflight wifi service',
            'Online boarding', 'Seat comfort', 'Inflight entertainment']
absent = [c for c in features if c not in df.columns]
if absent:
    raise ValueError(f'These columns are missing from the file: {absent}')

X_raw = df[features].copy()
print('Features used:', features)
print('Records:', len(X_raw), '| Features:', len(features))
print('Missing values in these features:', int(X_raw.isna().sum().sum()))

X_imp = SimpleImputer(strategy='median').fit_transform(X_raw)
scaler = StandardScaler()
X = scaler.fit_transform(X_imp)
print('After scaling: mean =', X.mean(axis=0).round(2), '| std =', X.std(axis=0).round(2))

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
ev = pca.explained_variance_ratio_ * 100
print(f'PCA (for plotting only): PC1 = {ev[0]:.1f}%, PC2 = {ev[1]:.1f}% of the variance')
display(X_raw.describe().T[['mean', 'std', 'min', 'max']].round(2))

## Helper functions
No screenshot needed. These only draw charts and calculate metrics.

In [ ]:
PALETTE = plt.cm.tab10.colors

def scatter_clusters(ax, pts, labels, title):
    for lab in sorted(set(labels)):
        m = labels == lab
        if lab == -1:
            ax.scatter(pts[m, 0], pts[m, 1], s=6, color='lightgrey', alpha=0.6, label='Noise')
        else:
            ax.scatter(pts[m, 0], pts[m, 1], s=6, color=PALETTE[int(lab) % 10], alpha=0.6, label=f'Cluster {lab}')
    ax.set_title(title); ax.set_xlabel('PCA component 1'); ax.set_ylabel('PCA component 2')
    ax.legend(markerscale=3)

def size_bar(ax, labels, title):
    s = pd.Series(labels).value_counts().sort_index()
    names = ['Noise' if k == -1 else f'Cluster {k}' for k in s.index]
    colors = ['lightgrey' if k == -1 else PALETTE[int(k) % 10] for k in s.index]
    ax.bar(names, s.values, color=colors)
    for i, v in enumerate(s.values):
        ax.text(i, v, f'{v:,}\n({v / len(labels) * 100:.1f}%)', ha='center', va='bottom')
    ax.set_ylim(0, s.max() * 1.25); ax.set_title(title); ax.set_ylabel('Records')

def cluster_metrics(data, labels):
    keep = labels != -1                       # noise points are left out
    d, l = data[keep], labels[keep]
    if len(set(l)) < 2:
        return np.nan, np.nan, np.nan
    sil = silhouette_score(d, l, sample_size=min(10000, len(l)), random_state=RANDOM_STATE)
    return sil, davies_bouldin_score(d, l), calinski_harabasz_score(d, l)

print('Helpers ready.')

## K-Means: choose the number of clusters — Screenshot 2
The elbow plot shows how the within-cluster distance falls as *k* grows. The silhouette score (higher is better) is also calculated for k = 2 to 8, and the k with the best silhouette is used. This cell may take about a minute.

In [ ]:
k_values = list(range(1, 11))
inertia = [KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X).inertia_ for k in k_values]

sil_scores = {}
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    sil_scores[k] = silhouette_score(X, lab, sample_size=10000, random_state=RANDOM_STATE)
best_k = max(sil_scores, key=sil_scores.get)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(k_values, inertia, marker='o', color='#1F4E79')
axes[0].set_title('Elbow Method'); axes[0].set_xlabel('Number of clusters (k)'); axes[0].set_ylabel('Inertia (within-cluster sum of squares)')
axes[1].plot(list(sil_scores), list(sil_scores.values()), marker='o', color='#C55A11')
axes[1].axvline(best_k, color='grey', linestyle='--', label=f'Best k = {best_k}')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('Number of clusters (k)'); axes[1].set_ylabel('Silhouette score'); axes[1].legend()
plt.tight_layout(); plt.show()
print('Silhouette score for each k:', {k: round(v, 3) for k, v in sil_scores.items()})
print(f'Best k = {best_k}')

## K-Means result — Screenshot 3

In [ ]:
K = best_k
t0 = time.perf_counter()
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(X)
km_time = time.perf_counter() - t0
km_labels = kmeans.labels_
km_sil, km_db, km_ch = cluster_metrics(X, km_labels)
print(f'K-Means with k = {K} on {len(X):,} records | time = {km_time:.2f} s')
print(f'Silhouette = {km_sil:.3f} | Davies-Bouldin = {km_db:.3f} | Calinski-Harabasz = {km_ch:,.0f}')

centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features).round(2)
centroids.index = [f'Cluster {i}' for i in range(K)]
centroids.insert(0, 'Records', np.bincount(km_labels))
display(centroids)

plot_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), 10000, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
scatter_clusters(axes[0], X_pca[plot_idx], km_labels[plot_idx], f'K-Means (k = {K}), 10,000 records shown')
heat = centroids.drop(columns='Records')
shade = (heat - heat.min()) / (heat.max() - heat.min())
sns.heatmap(shade, annot=heat.values, fmt='.2f', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set_title('Cluster centres (original units; shading is relative within each column)')
plt.tight_layout(); plt.show()

## Agglomerative clustering: dendrogram — Screenshot 4
Agglomerative clustering needs memory that grows with the square of the number of records, so a random sample is used: 200 records for the dendrogram and 3,000 for the clustering. The red line shows where the tree is cut to give the same number of clusters as K-Means.

In [ ]:
dendro_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), 200, replace=False)
Z = linkage(X[dendro_idx], method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=9, show_contracted=True)
cut = (Z[-K, 2] + Z[-(K - 1), 2]) / 2
plt.axhline(cut, color='red', linestyle='--', label=f'Cut for {K} clusters')
plt.legend()
plt.title('Dendrogram (Ward linkage, random sample of 200 records)')
plt.xlabel('Merged groups (number of records in brackets)'); plt.ylabel('Distance')
plt.tight_layout(); plt.show()

## Agglomerative clustering result — Screenshot 5
The table compares four linkage methods on the same 3,000 records. Ward linkage is used for the final result.

In [ ]:
agg_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), 3000, replace=False)
X_agg = X[agg_idx]

rows = []
for link in ['ward', 'complete', 'average', 'single']:
    t0 = time.perf_counter()
    lab = AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X_agg)
    rows.append({'Linkage': link, 'Silhouette': silhouette_score(X_agg, lab),
                 'Largest cluster (%)': np.bincount(lab).max() / len(lab) * 100,
                 'Time (s)': time.perf_counter() - t0})
display(pd.DataFrame(rows).set_index('Linkage').round(3))

t0 = time.perf_counter()
agg_labels = AgglomerativeClustering(n_clusters=K, linkage='ward').fit_predict(X_agg)
agg_time = time.perf_counter() - t0
agg_sil, agg_db, agg_ch = cluster_metrics(X_agg, agg_labels)
print(f'Agglomerative (Ward) with {K} clusters on {len(X_agg):,} records | time = {agg_time:.2f} s')
print(f'Silhouette = {agg_sil:.3f} | Davies-Bouldin = {agg_db:.3f} | Calinski-Harabasz = {agg_ch:,.0f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter_clusters(axes[0], X_pca[agg_idx], agg_labels, f'Agglomerative (Ward, {K} clusters)')
size_bar(axes[1], agg_labels, 'Cluster sizes')
plt.tight_layout(); plt.show()

## DBSCAN: choose epsilon — Screenshot 6
DBSCAN needs a radius (`eps`) and a minimum number of neighbours (`min_samples`). `min_samples` is set to 2 × the number of features. For `eps`, each point's distance to its 12th nearest neighbour is sorted and plotted; the bend (elbow) of the curve is a good value. A random sample of 10,000 records is used.

In [ ]:
db_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), 10000, replace=False)
X_db = X[db_idx]
min_samples = 2 * X.shape[1]

dist, _ = NearestNeighbors(n_neighbors=min_samples).fit(X_db).kneighbors(X_db)
kdist = np.sort(dist[:, -1])

# elbow = point farthest below the straight line joining the first and last points
xs = np.linspace(0, 1, len(kdist))
ys = (kdist - kdist.min()) / (kdist.max() - kdist.min())
eps = round(float(kdist[int(np.argmax(xs - ys))]), 2)      # you may type a different value here

plt.figure(figsize=(8, 4.5))
plt.plot(kdist, color='#1F4E79')
plt.axhline(eps, color='red', linestyle='--', label=f'Chosen eps = {eps}')
plt.title(f'Sorted distance to the {min_samples}th nearest neighbour')
plt.xlabel('Records (sorted)'); plt.ylabel('Distance'); plt.legend()
plt.tight_layout(); plt.show()
print(f'min_samples = {min_samples} | eps = {eps}')

## DBSCAN result — Screenshot 7
Grey points are noise (records that do not belong to any dense cluster).

In [ ]:
t0 = time.perf_counter()
db_labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_db)
db_time = time.perf_counter() - t0
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int((db_labels == -1).sum())
db_sil, db_db, db_ch = cluster_metrics(X_db, db_labels)

print(f'DBSCAN (eps = {eps}, min_samples = {min_samples}) on {len(X_db):,} records | time = {db_time:.2f} s')
print(f'Clusters found: {n_clusters_db} | Noise points: {n_noise:,} ({n_noise / len(db_labels) * 100:.1f}%)')
if n_clusters_db >= 2:
    print(f'Silhouette = {db_sil:.3f} | Davies-Bouldin = {db_db:.3f} | Calinski-Harabasz = {db_ch:,.0f}  (noise excluded)')
else:
    print('Silhouette, Davies-Bouldin and Calinski-Harabasz are not defined: fewer than 2 clusters found.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter_clusters(axes[0], X_pca[db_idx], db_labels, f'DBSCAN (eps = {eps}, min_samples = {min_samples})')
size_bar(axes[1], db_labels, 'Cluster sizes (including noise)')
plt.tight_layout(); plt.show()

## Compare the three algorithms — Screenshot 8
The three algorithms were run on different numbers of records (K-Means on all records, Agglomerative on 3,000, DBSCAN on 10,000), and DBSCAN's metrics leave out noise points, so compare the metrics with care.

In [ ]:
comparison = pd.DataFrame({
    'K-Means': {'Records used': len(X), 'Clusters': K, 'Noise points': 0,
                'Silhouette': km_sil, 'Davies-Bouldin': km_db, 'Calinski-Harabasz': km_ch, 'Time (s)': km_time},
    'Agglomerative (Ward)': {'Records used': len(X_agg), 'Clusters': K, 'Noise points': 0,
                'Silhouette': agg_sil, 'Davies-Bouldin': agg_db, 'Calinski-Harabasz': agg_ch, 'Time (s)': agg_time},
    'DBSCAN': {'Records used': len(X_db), 'Clusters': n_clusters_db, 'Noise points': n_noise,
                'Silhouette': db_sil, 'Davies-Bouldin': db_db, 'Calinski-Harabasz': db_ch, 'Time (s)': db_time},
}).T
comparison[['Records used', 'Clusters', 'Noise points']] = comparison[['Records used', 'Clusters', 'Noise points']].astype('Int64')
display(comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
comparison['Silhouette'].dropna().astype(float).plot(kind='bar', ax=axes[0], color='#1F4E79')
axes[0].set_title('Silhouette (higher is better)'); axes[0].tick_params(axis='x', rotation=15)
comparison['Davies-Bouldin'].dropna().astype(float).plot(kind='bar', ax=axes[1], color='#C55A11')
axes[1].set_title('Davies-Bouldin (lower is better)'); axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

## Result
Three clustering algorithms were applied to six standardised numeric features of the airline passenger data. Record the number of clusters, the cluster sizes, the silhouette and Davies-Bouldin values and the DBSCAN noise percentage from the outputs above, and state which algorithm gave the most useful grouping. K-Means needs *k* in advance, Agglomerative clustering shows how clusters merge but is slow on large data, and DBSCAN finds clusters of any shape and labels noise but depends strongly on `eps`.